In [1]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 

emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [2]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [3]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


### Design of the Viterbi Value matrix

In [ ]:
# Initiate two matrices

num_states = len(states)
seq_len = len(query_sequence)

viterbi_value_matrix = np.full((num_states, seq_len), -np.inf)

viterbi_trace_matrix = np.full((num_states, seq_len), -1)

first_nucleotide = query_sequence[0]

for current_state in range(num_states):

    transition_prob = state_transition_prob[
        states['s']
    ][
        current_state
    ]

    emission_prob = emission_probs[
        current_state
    ][
        emission_nuc_codes[first_nucleotide]
    ]

    if transition_prob > 0 and emission_prob > 0:

        viterbi_value_matrix[current_state][0] = (
            math.log(transition_prob) +
            math.log(emission_prob)
        )

    viterbi_trace_matrix[current_state][0] = states['s']


### Implementation of Viterbi Algorithm

In [ ]:
def calculate_prob_for_a_node(
    current_state,
    column,
    query_sequence,
    viterbi_value_matrix
):

    nucleotide = query_sequence[column]

    emission_prob = emission_probs[current_state][
        emission_nuc_codes[nucleotide]
    ]

    if emission_prob == 0:
        return -np.inf, -1

    best_value = -np.inf
    best_previous_state = -1

    for previous_state in range(num_states):

        transition_prob = state_transition_prob[
            previous_state
        ][
            current_state
        ]

        if transition_prob == 0:
            continue

        previous_value = viterbi_value_matrix[
            previous_state
        ][
            column - 1
        ]

        if previous_value == -np.inf:
            continue

        candidate_value = (
            previous_value +
            math.log(transition_prob) +
            math.log(emission_prob)
        )

        if candidate_value > best_value:
            best_value = candidate_value
            best_previous_state = previous_state

    return best_value, best_previous_state


In [ ]:
for column in range(1, seq_len):

    for current_state in range(num_states):

        best_value, best_previous_state = calculate_prob_for_a_node(
            current_state,
            column,
            query_sequence,
            viterbi_value_matrix
        )

        viterbi_value_matrix[current_state][column] = best_value

        viterbi_trace_matrix[current_state][column] = best_previous_state

print('Viterbi Value Matrix')
print(viterbi_value_matrix)

print('\nViterbi Trace Matrix')
print(viterbi_trace_matrix)


In [ ]:
# traceback

last_column = viterbi_value_matrix[:, seq_len - 1]

best_final_state = np.argmax(last_column)

best_path = [best_final_state]

current_state = best_final_state

for column in range(seq_len - 1, 0, -1):

    previous_state = viterbi_trace_matrix[
        current_state
    ][
        column
    ]

    best_path.append(previous_state)

    current_state = previous_state

best_path.reverse()

decoded_states = ''

for state_id in best_path:
    decoded_states += id2state[state_id]

print('\nMost Probable State Path')
print(decoded_states)
